<a href="https://colab.research.google.com/github/LinglinNero/MSSP607/blob/main/PARTICIPATION_ACTIVITY_Week_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#connect to github file
from google.colab import userdata
import os

github_token = userdata.get('Git_Hub')
owner = 'LinglinNero' # Replace with the GitHub repository owner
repository = 'MSSP607' # Replace with the GitHub repository name

clone_url = f'https://{github_token}@github.com/{owner}/{repository}.git'

# Clone the repository
!git clone {clone_url}

fatal: destination path 'MSSP607' already exists and is not an empty directory.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import requests
import sys
import os
import urllib.parse
import re
import mimetypes
from bs4 import BeautifulSoup

page_url = 'https://www.sp2.upenn.edu/people/faculty/richard-harris-hartwell/'
save_folder = os.path.join('.','image_search', 'richard-h-hartwell-edd')
os.makedirs(save_folder, exist_ok=True)

headers = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'),
    'Referer': page_url,
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
}

def choose_srcset(srcset: str) -> str | None:
  if not srcset:
    return None
  parts = [p.strip() for p in srcset.split(',') if p.strip()]
  if not parts:
    return None
  last = parts[-1]
  url = last.split()[0]
  return url

def infer_filename(url: str, resp: requests.Response | None) -> str:
  path = urllib.parse.urlparse(url).path
  fname = os.path.basename(path)
  if (not fname) or ('.' not in fname):
    ext = None
    if resp is not None:
      ctype = resp.headers.get('Content-Type', '').split(';')[0].strip()
      ext = mimetypes.guess
    if not ext:
      ext = '.jpg'
    base = re.sub(r'\W+', '_', os.path.splitext(fname or 'image')[0]).strip('_') or 'image'
    fname = base + ext
  return fname

def save_image(img_url: str, index: int):
  abs_url = urllib.parse.urljoin(page_url, img_url)
  with requests.get(abs_url, headers=headers, stream=True, timeout=20) as r:
    r.raise_for_status()
    filename = infer_filename(abs_url, r)
    name, ext = os.path.splitext(filename)
    outfile = os.path.join(save_folder, f'{index:03d}_{name}{ext}')
    with open(outfile, 'wb') as f:
      for chunk in r.iter_content(chunk_size=1024 * 64):
        if chunk:
          f.write(chunk)
    print(f'[OK] {abs_url} -> {outfile}')


try:
  res = requests.get(page_url, headers = headers)
  res.raise_for_status()
except requests.exceptions.RequestException as err:
  print(f'cannot download {page_url},error: {err}')
  sys.exit(1)

soup = BeautifulSoup(res.text, 'html.parser')
candidates = []

for img in soup.find_all('img'):
  src = (img.get('src') or img.get('data-src') or img.get('data-lazy-src'))
  if not src:
    srcset = img.get('srcset')
    src = choose_srcset(srcset)
  if src:
    candidates.append(src)

open_graph = soup.find('meta', property='og:image')
if open_graph and open_graph.get('content'):
  candidates.append(open_graph['content'])

allow_ext = ('.jpg', '.jpeg', '.png', '.webp', '.gif', '.bmp', '.svg')

def maybe_image(u: str) -> bool:
  path = urllib.parse.urlparse(u).path.lower()
  if any(path.endswith(ext) for ext in allow_ext):
    return True
  return True

seen = set()
filtered = []
for u in candidates:
  if not u:
    continue
  u = u.strip()
  if not u or u.startswith('data:'):
    continue
  if u in seen:
    continue
  if maybe_image(u):
    seen.add(u)
    filtered.append(u)



if not filtered:
  print('could not find any images')
else:
  print(f'found {len(filtered)} candidate images, downloading')
  for i, u in enumerate(filtered, 1):
    try:
      save_image(u, i)
    except requests.exceptions.RequestException as err:
      print(f'[SKIP] {u} error: {err}')
print('done！')






found 21 candidate images, downloading
[OK] https://sp2.upenn.edu/wp-content/uploads/2020/02/Elizabeth-Abel_1-150x150.jpg -> ./image_search/richard-h-hartwell-edd/001_Elizabeth-Abel_1-150x150.jpg
[OK] https://sp2.upenn.edu/wp-content/uploads/2021/07/Millan_AbiNader-150x150.jpg -> ./image_search/richard-h-hartwell-edd/002_Millan_AbiNader-150x150.jpg
[OK] https://sp2.upenn.edu/wp-content/uploads/2014/07/Jane-Abrams-150x150.jpg -> ./image_search/richard-h-hartwell-edd/003_Jane-Abrams-150x150.jpg
[OK] https://sp2.upenn.edu/wp-content/uploads/2020/02/Ginneh-Akbar-150x150.jpg -> ./image_search/richard-h-hartwell-edd/004_Ginneh-Akbar-150x150.jpg
[OK] https://sp2.upenn.edu/wp-content/uploads/2014/07/Valerie_Dorsey_Allen-150x150.jpg -> ./image_search/richard-h-hartwell-edd/005_Valerie_Dorsey_Allen-150x150.jpg
[OK] https://sp2.upenn.edu/wp-content/uploads/2023/07/Gina-Amoroso-Latta-150x150.jpg -> ./image_search/richard-h-hartwell-edd/006_Gina-Amoroso-Latta-150x150.jpg
[OK] https://sp2.upenn.edu/